# 1. Generate naive forecasts

For every eligible 5-minute forecast origin, copy the complete previous 48-hour observed price curve forward by 48 hours. This produces a varying, strictly causal 96-point forecast (30 minutes × 48 hours). The archive is generated before choosing an operational forecast cadence or overlap policy.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../../"))
from EPF import variables

HORIZON_LIST = range(1, variables.HORIZON_COUNT + 1)

In [2]:
# Build the observed 30-minute price at every 5-minute timestamp using the
# same six-price trailing mean as the model target construction.
observations_per_target = variables.FEATURE_STEPS_PER_HORIZON
spot_price = pd.read_parquet(variables.TARGET_DATASET_PATH, columns=[variables.SELECTED_TARGET_COLUMN_NAME])[variables.SELECTED_TARGET_COLUMN_NAME].astype(np.float32)
observed_30min_price = spot_price.rolling(observations_per_target, min_periods=observations_per_target).mean()

In [3]:
# Use exactly the origins for which a complete 48-hour target path exists.
target_columns = [f"target_h{horizon}" for horizon in HORIZON_LIST]

forecast_origins = (
    pd.read_parquet(variables.AGG_TARGET_DATASET_PATH, columns=target_columns)
    .dropna(subset=target_columns)
    .index
)

# Copy the observed 48-hour block ending at each origin onto the next 48
# hours. For future horizon h, use the corresponding delivery block from
# exactly 48 hours earlier. h96 therefore uses the block ending at the origin,
# so no prediction uses information from after the forecast origin.
lookback = pd.Timedelta(hours=variables.HORIZON_LENGTH_IN_HOURS)
naive_matrix = np.empty((len(forecast_origins), variables.HORIZON_COUNT), dtype=np.float32)
for column, horizon in enumerate(HORIZON_LIST):
    historical_delivery_end = (
        forecast_origins
        - lookback
        + pd.Timedelta(minutes=horizon * variables.HORIZON_GRANULARITY_IN_MINUTES)
    )
    naive_matrix[:, column] = observed_30min_price.reindex(historical_delivery_end).to_numpy(dtype=np.float32)

naive_forecasts = pd.DataFrame(
    naive_matrix,
    index=forecast_origins,
    columns=[f"predicted_h{horizon}" for horizon in HORIZON_LIST],
)
naive_forecasts.index.name = "forecast_origin"

variables.MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
naive_forecasts.to_parquet(variables.NAIVE_FORECASTS_PATH)
print(f"Saved {len(naive_forecasts):,} complete 48-hour block-persistence forecasts to {variables.NAIVE_FORECASTS_PATH}")
naive_forecasts.head()

Saved 736,417 complete 48-hour block-persistence forecasts to C:\Users\Daniel\OneDrive\Documents\2_Energy_toolkit\1_ST_price_prediction\EPF\5_Model\Data\5_model_results\naive_forecasts.parquet


,predicted_h1,predicted_h2,predicted_h3,predicted_h4,predicted_h5,predicted_h6,predicted_h7,predicted_h8,predicted_h9,predicted_h10,...,predicted_h87,predicted_h88,predicted_h89,predicted_h90,predicted_h91,predicted_h92,predicted_h93,predicted_h94,predicted_h95,predicted_h96
forecast_origin,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00,64.743111,69.068321,69.518677,66.509155,65.585365,63.623810,61.049355,52.733269,61.456402,55.013962,...,65.440186,68.124741,80.296829,83.516052,92.173294,79.032288,77.893517,78.064590,71.249268,70.107643
2019-01-01 00:05:00,65.610390,67.849373,70.342880,67.759178,62.482315,64.679581,59.394466,52.128723,61.546417,56.280266,...,65.762390,70.170639,81.098587,82.720047,93.449043,77.020248,79.194237,76.333481,71.560989,68.978027
2019-01-01 00:10:00,66.373726,68.326294,68.872398,67.705254,61.940842,64.415779,58.214542,54.017532,58.641136,58.512253,...,66.194298,71.998238,81.829147,83.990234,91.517601,76.250023,80.127205,74.854858,71.664978,68.356277
2019-01-01 00:15:00,66.774803,68.392990,67.785446,67.865501,61.744228,63.947269,56.984612,55.852417,56.700413,59.865555,...,66.806526,73.849907,83.162476,86.790237,86.570518,76.418564,80.767090,73.592041,71.766449,67.567978
2019-01-01 00:20:00,68.142235,68.264626,67.136414,67.315887,62.186829,63.342445,56.048168,56.783226,56.330593,59.881611,...,67.216583,75.943222,83.409348,88.846725,84.932632,75.589783,80.001030,72.857407,71.337143,67.549721
